In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from gridworld import Gridworld, random_mdp, build_vocab, tokenize_trace, detokenize

In [4]:
# Load the saved dataset and tokens
with open('dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

with open('tokenized.pkl', 'rb') as f:
    tokenized_data = pickle.load(f)

# Confirm GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Vocab size: {len(tokenized_data['vocab'])}")
print(f"Number of traces: {len(tokenized_data['tokens'])}")

Using device: cuda
Vocab size: 12
Number of traces: 600


In [5]:
def build_vocab_with_pad(max_grid_size, actions):
    vocab = {}
    vocab['<pad>'] = len(vocab)  # id 0
    for i in range(max_grid_size):
        vocab[f'coord_{i}'] = len(vocab)
    for a in actions:
        vocab[f'action_{a}'] = len(vocab)
    vocab['reward_step'] = len(vocab)
    vocab['reward_goal'] = len(vocab)
    vocab['<eos>'] = len(vocab)
    return vocab

vocab = build_vocab_with_pad(max_grid_size=5, actions=['up', 'down', 'left', 'right'])
print(f"New vocab size: {len(vocab)}")
print(f"Pad id: {vocab['<pad>']}")

# Re-tokenize all traces against the new vocab
tokenized = [tokenize_trace(entry['trace'], vocab) for entry in dataset['traces']]
mdp_indices = [entry['mdp_idx'] for entry in dataset['traces']]

New vocab size: 13
Pad id: 0


In [6]:
all_mdp_indices = list(range(10))
rng = np.random.default_rng(0)  # fixed seed for reproducibility
val_mdp_indices = sorted(rng.choice(all_mdp_indices, size=2, replace=False).tolist())
train_mdp_indices = [i for i in all_mdp_indices if i not in val_mdp_indices]

print(f"Train MDPs: {train_mdp_indices}")
print(f"Val MDPs: {val_mdp_indices}")

Train MDPs: [0, 1, 2, 3, 4, 5, 8, 9]
Val MDPs: [6, 7]


In [7]:
class TraceDataset(Dataset):
    def __init__(self, tokenized_traces, mdp_indices, allowed_mdp_indices, context_length, pad_id):
        # filter to traces from allowed MDPs
        self.traces = [
            tokens for tokens, mdp_idx in zip(tokenized_traces, mdp_indices)
            if mdp_idx in allowed_mdp_indices and len(tokens) <= context_length
        ]
        self.context_length = context_length
        self.pad_id = pad_id
    
    def __len__(self):
        return len(self.traces)
    
    def __getitem__(self, idx):
        tokens = self.traces[idx]
        # pad to context_length
        padded = tokens + [self.pad_id] * (self.context_length - len(tokens))
        padded = torch.tensor(padded, dtype=torch.long)
        # input is everything except last position; target is everything except first
        input_ids = padded[:-1]
        target_ids = padded[1:]
        return input_ids, target_ids

In [8]:
context_length = 128
pad_id = vocab['<pad>']

train_dataset = TraceDataset(tokenized, mdp_indices, set(train_mdp_indices), context_length, pad_id)
val_dataset = TraceDataset(tokenized, mdp_indices, set(val_mdp_indices), context_length, pad_id)

print(f"Train traces: {len(train_dataset)}")
print(f"Val traces: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Train traces: 409
Val traces: 97


In [9]:
input_ids, target_ids = next(iter(train_loader))
print(f"Input shape: {input_ids.shape}")
print(f"Target shape: {target_ids.shape}")
print(f"First input sequence: {input_ids[0]}")
print(f"First target sequence: {target_ids[0]}")

Input shape: torch.Size([32, 127])
Target shape: torch.Size([32, 127])
First input sequence: tensor([ 1,  1,  9, 10,  1,  2,  1,  2,  9, 10,  1,  3,  1,  3,  9, 10,  1,  4,
         1,  4,  7, 11,  2,  4, 12,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0])
First target sequence: tensor([ 1,  9, 10,  1,  2,  1,  2,  9, 10,  1,  3,  1,  3,  9, 10,  1,  4,  1,
         4,  7, 11,  2,  4, 12,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,

In [10]:
# We need to track this to debug — let's verify
# Reconstruct which MDPs are in train and which trace this is
train_traces_with_mdp = [
    (tokens, mdp_idx) for tokens, mdp_idx in zip(tokenized, mdp_indices)
    if mdp_idx in set(train_mdp_indices) and len(tokens) <= context_length
]
print(f"First train trace's MDP idx: {train_traces_with_mdp[0][1]}")
print(f"That MDP's goal: {dataset['mdps'][train_traces_with_mdp[0][1]]['goal']}")

First train trace's MDP idx: 0
That MDP's goal: (2, 0)


In [11]:
for idx in train_mdp_indices:
    print(f"MDP {idx}: goal={dataset['mdps'][idx]['goal']}")

MDP 0: goal=(2, 0)
MDP 1: goal=(1, 0)
MDP 2: goal=(1, 3)
MDP 3: goal=(2, 2)
MDP 4: goal=(2, 3)
MDP 5: goal=(2, 1)
MDP 8: goal=(3, 2)
MDP 9: goal=(1, 2)


In [14]:
class GridworldTransformer(nn.Module):
    def __init__(self, vocab_size, context_length, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, pad_id=0):
        super().__init__()
        self.d_model = d_model
        self.pad_id = pad_id
        self.context_length = context_length
        
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(context_length, d_model)
        
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        
        self.output_projection = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        # input_ids: (batch, seq_len)
        seq_len = input_ids.size(1)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # Causal mask: prevents attending to future positions
        causal_mask = torch.triu(
        torch.ones(seq_len, seq_len, device=input_ids.device, dtype=torch.bool),
        diagonal=1
        )
        
        # Padding mask: prevents attending to pad tokens
        pad_mask = (input_ids == self.pad_id)
        
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=pad_mask)
        logits = self.output_projection(x)
        return logits

In [15]:
model = GridworldTransformer(
    vocab_size=len(vocab),
    context_length=context_length,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    pad_id=pad_id,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Forward pass on a batch
input_ids, target_ids = next(iter(train_loader))
input_ids = input_ids.to(device)
target_ids = target_ids.to(device)

with torch.no_grad():
    logits = model(input_ids)

print(f"Logits shape: {logits.shape}")  # should be (32, 127, 13)

Total parameters: 76,813
Logits shape: torch.Size([32, 127, 13])


In [16]:
import torch.nn.functional as F

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_tokens = 0
    
    for input_ids, target_ids in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        # Forward
        logits = model(input_ids)
        # logits: (batch, seq_len, vocab_size)
        # targets: (batch, seq_len)
        # Reshape for cross-entropy
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),  # (batch * seq_len, vocab_size)
            target_ids.reshape(-1)                 # (batch * seq_len,)
        )
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track loss (weighted by non-pad tokens for accurate average)
        non_pad = (target_ids != pad_id).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    
    return total_loss / total_tokens

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    
    for input_ids, target_ids in loader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)
        
        logits = model(input_ids)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            target_ids.reshape(-1)
        )
        
        non_pad = (target_ids != pad_id).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad
    
    return total_loss / total_tokens

In [17]:
num_epochs = 20

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}: train loss = {train_loss:.4f}, val loss = {val_loss:.4f}")

Epoch 1: train loss = 2.1250, val loss = 1.7072
Epoch 2: train loss = 1.5661, val loss = 1.3891
Epoch 3: train loss = 1.3007, val loss = 1.1548
Epoch 4: train loss = 1.1151, val loss = 1.0009
Epoch 5: train loss = 0.9871, val loss = 0.8992
Epoch 6: train loss = 0.9013, val loss = 0.8300
Epoch 7: train loss = 0.8341, val loss = 0.7841
Epoch 8: train loss = 0.7828, val loss = 0.7365
Epoch 9: train loss = 0.7340, val loss = 0.6742
Epoch 10: train loss = 0.6801, val loss = 0.6296
Epoch 11: train loss = 0.6443, val loss = 0.5929
Epoch 12: train loss = 0.6125, val loss = 0.5518
Epoch 13: train loss = 0.5860, val loss = 0.5389
Epoch 14: train loss = 0.5618, val loss = 0.5137
Epoch 15: train loss = 0.5412, val loss = 0.4970
Epoch 16: train loss = 0.5255, val loss = 0.4808
Epoch 17: train loss = 0.5123, val loss = 0.4828
Epoch 18: train loss = 0.4958, val loss = 0.4571
Epoch 19: train loss = 0.4799, val loss = 0.4654
Epoch 20: train loss = 0.4719, val loss = 0.4480


In [18]:
# Check val MDP characteristics vs train MDP characteristics
for label, indices in [('train', train_mdp_indices), ('val', val_mdp_indices)]:
    sizes = [(dataset['mdps'][i]['size_x'], dataset['mdps'][i]['size_y']) for i in indices]
    slips = [dataset['mdps'][i]['slip_prob'] for i in indices]
    print(f"{label} MDPs:")
    for i, (s, sl) in zip(indices, zip(sizes, slips)):
        print(f"  MDP {i}: {s[0]}x{s[1]}, slip={sl:.3f}")
    print(f"  avg slip: {np.mean(slips):.3f}")

train MDPs:
  MDP 0: 5x3, slip=0.042
  MDP 1: 3x5, slip=0.217
  MDP 2: 3x5, slip=0.194
  MDP 3: 4x3, slip=0.049
  MDP 4: 5x5, slip=0.270
  MDP 5: 5x3, slip=0.172
  MDP 8: 5x5, slip=0.249
  MDP 9: 3x5, slip=0.300
  avg slip: 0.186
val MDPs:
  MDP 6: 4x3, slip=0.024
  MDP 7: 4x5, slip=0.297
  avg slip: 0.161


In [19]:
torch.save(model.state_dict(), 'model_v1.pt')